JSON dataset
      ↓
Load dataset
      ↓
Train / validation / test split
      ↓
Load tokenizer
      ↓
Load Qwen model
      ↓
Inspect chat template
      ↓
Format conversations
      ↓
Tokenization
      ↓
Inspect labels / loss masking
      ↓
LoRA
      ↓
SFT
      ↓
Evaluation

In [28]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


In [29]:
import json
from collections import Counter

with open("/content/electronics_tutoring_dataset.json", "r") as file:
    data = json.load(file)

print(len(data))
print("Kirchhoff law number:",sum(row["topic"] == "Kirchhoff's Laws" for row in data))
print("diodes number:",sum(row["topic"] == "Diodes" for row in data))
print("BJTs number:",sum(row["topic"] == "BJTs" for row in data))
print("MOSFETs number:",sum(row["topic"] == "MOSFETs" for row in data))
print("Op-amps number:",sum(row["topic"] == "Op-amps" for row in data))
print(" ")


required = ["id","topic","difficulty","instruction","question","answer"]

count = 0
for item in data:
  for field in required:
    if not item.get(field):
      count += 1

print(count)


250
Kirchhoff law number: 51
diodes number: 50
BJTs number: 50
MOSFETs number: 50
Op-amps number: 49
 
0


In [30]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(data,test_size=0.2,random_state=0,shuffle=True)

valid, test = train_test_split(temp,test_size=0.5,random_state=0,shuffle=True)


print("Train:", len(train))
print("Valid:", len(valid))
print("Test:", len(test))

Train: 200
Valid: 25
Test: 25


In [31]:
print(train[0])

{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}


In [32]:
from transformers import AutoModelForCausalLM, AutoTokenizer


checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [33]:
model.num_parameters()

1543714304

In [34]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(checkpoint)

In [35]:
print(config)

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [36]:
print("vocab size",len(tokenizer))
print("special tokens",tokenizer.special_tokens_map)
print("")
if tokenizer.chat_template:
  print(tokenizer.chat_template)

vocab size 151665
special tokens {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end

In [37]:

text = train[0]
print(text)
instruction = text["instruction"]
question = text["question"]
answer = text["answer"]


messages = [
    {
        "role" : "system",
        "content" : instruction
    },
    {
        "role" : "user",
        "content":question,
    },
    {
        "role" : "assistant",
        "content":answer
    }
]

to_chat_template = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt = False)
print(to_chat_template)
print("")

tokenized_message = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]
print(tokenized_message)

decoded_message = tokenizer.decode(tokenized_message)
print(decoded_message)



{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}
<|im_start|>system
Explain this concept as if you were tutoring an undergraduate engineering student.<|im_end|>
<|im_start|>user
Why does a diode conduct current in only one direction?<|im_end|>
<|im_start|>assistant
The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction.<|im_end|>


[151

In [38]:
print(tokenizer("<|im_start|>assistant",add_special_tokens=False)["input_ids"])
print(tokenizer("<|im_end|>",add_special_tokens=False)["input_ids"])

[151644, 77091]
[151645]


In [39]:
tokens = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]
for i,token in enumerate(tokens):
  if(tokens[i] == 151644 and tokens[i+1] == 77091):
    start = i+2

for j in range(start,len(tokens)):
  if(tokens[j] == 151645):
    end = j

print(start)
print(end)



38
83


In [40]:
labels = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]

for i in range(0,start):
  labels[i] = -100

for j in range(end,len(labels)):
  labels[j] = -100

print(len(tokens))
print(len(labels))
print((tokens[34:84]))
print((labels[34:84]))


85
85
[151645, 198, 151644, 77091, 198, 785, 61901, 48241, 594, 91848, 5537, 323, 5798, 3419, 9072, 2070, 33034, 1482, 6396, 304, 9931, 15470, 553, 22188, 8686, 18602, 7203, 11, 1393, 304, 4637, 15470, 279, 9251, 21720, 916, 6579, 419, 22103, 323, 6147, 8686, 34891, 311, 6396, 3941, 279, 48241, 13, 151645]
[-100, -100, -100, -100, 198, 785, 61901, 48241, 594, 91848, 5537, 323, 5798, 3419, 9072, 2070, 33034, 1482, 6396, 304, 9931, 15470, 553, 22188, 8686, 18602, 7203, 11, 1393, 304, 4637, 15470, 279, 9251, 21720, 916, 6579, 419, 22103, 323, 6147, 8686, 34891, 311, 6396, 3941, 279, 48241, 13, -100]


In [41]:
print(model.named_parameters)

<bound method Module.named_parameters of Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

In [42]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [43]:
from peft import get_peft_model,LoraConfig

peft_config = LoraConfig(
    target_modules = ["q_proj","v_proj"],
    task_type = "CAUSAL_LM",
    r = 8,
    lora_alpha = 16,
    lora_dropout= 0.05
)

peft_model = get_peft_model(model,peft_config)
peft_model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [44]:
print(train)

[{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}, {'id': 161, 'topic': 'MOSFETs', 'difficulty': 'basic', 'instruction': 'Give a clear conceptual explanation of the following electronics question.', 'question': 'What is the body effect in a MOSFET?', 'answer': 'The body effect is the change in threshold voltage that occurs when there is a voltage difference between the source and the body terminal, which effectively makes the threshold voltage a function of the source-body voltage rather than a fixed constant.'}, {'id': 181, 'topic': 'MO

In [45]:
messages_train = []

for text in train:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_train.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

In [46]:
print(len(messages_train))
print(messages_train[0])

200
[{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': 'Why does a diode conduct current in only one direction?'}, {'role': 'assistant', 'content': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}]


In [47]:
messages_validation = []

for text in valid:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_validation.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

In [48]:
print(len(messages_validation))
print(messages_validation[0])

25
[{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': "Why doesn't a MOSFET's gate draw significant current in steady state?"}, {'role': 'assistant', 'content': 'The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.'}]


In [49]:
from datasets import Dataset

# Convert each conversation into {"messages": [...]}
train_data = [{"messages": conversation} for conversation in messages_train]

train_dataset = Dataset.from_list(train_data)

print(len(train_dataset))
print(train_dataset[0])

200
{'messages': [{'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'role': 'system'}, {'content': 'Why does a diode conduct current in only one direction?', 'role': 'user'}, {'content': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction.", 'role': 'assistant'}]}


In [50]:
valid_data = [{"messages": conversation} for conversation in messages_validation]

valid_dataset = Dataset.from_list(valid_data)

print(len(valid_dataset))
print(valid_dataset[0])

25
{'messages': [{'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'role': 'system'}, {'content': "Why doesn't a MOSFET's gate draw significant current in steady state?", 'role': 'user'}, {'content': 'The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.', 'role': 'assistant'}]}


In [51]:
!pip install -U trl

In [52]:
import trl
print(trl.__version__)

1.10.0


In [53]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [54]:
from trl import SFTConfig,SFTTrainer

sft_config = SFTConfig(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    # Sequence length
    max_length=512,

    # Evaluation
    eval_strategy="epoch",

    # Saving
    save_strategy="epoch",

    # Train only on assistant responses
    assistant_only_loss=True,

    # Optional
    logging_steps=10,
    report_to="none",
)

trainer = SFTTrainer(model=peft_model,args=sft_config,train_dataset=train_dataset,eval_dataset=valid_dataset)

trainer.train()

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.724004,1.668288,1.531764,18374.000000,0.581605
2,1.544146,1.581987,1.507529,36748.000000,0.593918
3,1.476655,1.554615,1.480311,55122.000000,0.599394


TrainOutput(global_step=75, training_loss=1.6062900479634603, metrics={'train_runtime': 335.1806, 'train_samples_per_second': 1.79, 'train_steps_per_second': 0.224, 'total_flos': 486498617659392.0, 'train_loss': 1.6062900479634603, 'epoch': 3.0})

In [55]:
print(trainer.model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [56]:
trainer.save_model("./electronics-lora")
tokenizer.save_pretrained("./electronics-lora")

('./electronics-lora/tokenizer_config.json',
 './electronics-lora/chat_template.jinja',
 './electronics-lora/tokenizer.json')

In [57]:
import os

print(os.listdir("./electronics-lora"))

['README.md', 'adapter_model.safetensors', 'training_args.bin', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json', 'tokenizer.json']


Evaluation

A)Base Model (Qwen2.5-1.5B-Instruct)

B) Fine-tuned model (Qwen2.5-1.5B-Instruct + LoRA)

In [58]:
messages_test = []

for text in test:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_test.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

from datasets import Dataset

# Convert each conversation into {"messages": [...]}
test_data = [{"messages": conversation} for conversation in messages_test]

test_data = Dataset.from_list(test_data)

print(len(test_data))
print(test_data[0])

25
{'messages': [{'content': 'Explain this electronics concept to an undergraduate student.', 'role': 'system'}, {'content': "What happens to a MOSFET's drain current if the drain-source voltage is increased well beyond the overdrive voltage while gate-source voltage stays fixed?", 'role': 'user'}, {'content': 'Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.', 'role': 'assistant'}]}


In [59]:
test_sample1 = test_data[0]

instruction = test_sample1["messages"][0]["content"]
question = test_sample1["messages"][1]["content"]
reference_answer = test_sample1["messages"][2]["content"]

print("INSTRUCTION:")
print(instruction)

print("\nQUESTION:")
print(question)

print("\nREFERENCE ANSWER:")
print(reference_answer)

INSTRUCTION:
Explain this electronics concept to an undergraduate student.

QUESTION:
What happens to a MOSFET's drain current if the drain-source voltage is increased well beyond the overdrive voltage while gate-source voltage stays fixed?

REFERENCE ANSWER:
Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.


In [60]:
conversation = [
    {
        "role": "system",
        "content": instruction
    },
    {
        "role": "user",
        "content": question
    }
]

In [61]:
prompt = tokenizer.apply_chat_template(conversation,tokenize=False,add_generation_prompt=True)
print(prompt)

<|im_start|>system
Explain this electronics concept to an undergraduate student.<|im_end|>
<|im_start|>user
What happens to a MOSFET's drain current if the drain-source voltage is increased well beyond the overdrive voltage while gate-source voltage stays fixed?<|im_end|>
<|im_start|>assistant



In [62]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = peft_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("MODEL ANSWER:")
print(generated)

print("\nREFERENCE ANSWER:")
print(reference_answer)

MODEL ANSWER:
The drain current increases exponentially as long as it remains below the breakdown voltage, but once that threshold is exceeded, the device will be damaged and may fail catastrophically.

REFERENCE ANSWER:
Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.


In [63]:
checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(checkpoint)
base_tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [64]:
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

generated = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("MODEL ANSWER:")
print(generated)

print("\nREFERENCE ANSWER:")
print(reference_answer)

MODEL ANSWER:
When you increase the drain-source voltage (VDS) of a MOSFET past its overdrive voltage, several things happen:

1. **Threshold Voltage Increase**: The threshold voltage (VT), which is the minimum gate-to-source voltage required for the device to start conducting, increases. This means that it takes more positive gate voltage to turn on the MOSFET.

2. **Drain Current Increase**: As VDS increases further, the drain current (ID) also increases. However, there are practical limits to how much ID can be sustained before other factors come into play such as saturation region behavior or thermal considerations.

3. **Saturation Region**: Beyond a certain point, the drain current starts to saturate, meaning it approaches a maximum value determined by the material properties and geometry of the MOSFET. At this point, increasing VDS no longer significantly increases ID.

4. **Power Dissipation**: If the power dissipated in the MOSFET exceeds its rated power capability, it will ev

In [65]:
print(test[0])

{'id': 195, 'topic': 'MOSFETs', 'difficulty': 'applied', 'instruction': 'Explain this electronics concept to an undergraduate student.', 'question': "What happens to a MOSFET's drain current if the drain-source voltage is increased well beyond the overdrive voltage while gate-source voltage stays fixed?", 'answer': 'Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.'}


In [66]:
import torch

def calculate_test_loss(model, test_data, tokenizer, max_length=512):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for sample in test_data:

        # Create the conversation WITHOUT the answer
        messages = [
            {
                "role": "system",
                "content": sample["instruction"]
            },
            {
                "role": "user",
                "content": sample["question"]
            }
        ]

        # Prompt ending at assistant generation
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Full conversation INCLUDING the correct answer
        full_messages = messages + [
            {
                "role": "assistant",
                "content": sample["answer"]
            }
        ]

        full_text = tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        # Tokenize prompt and full conversation
        prompt_tokens = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        full_tokens = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        input_ids = full_tokens["input_ids"].to(model.device)
        attention_mask = full_tokens["attention_mask"].to(model.device)

        # Labels start as -100, meaning "don't calculate loss"
        labels = input_ids.clone()

        # Ignore everything belonging to system + user prompt
        prompt_length = prompt_tokens["input_ids"].shape[1]

        labels[:, :prompt_length] = -100

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

        # Count actual answer tokens
        num_answer_tokens = (labels != -100).sum().item()

        total_loss += outputs.loss.item() * num_answer_tokens
        total_tokens += num_answer_tokens

    return total_loss / total_tokens

In [78]:
lora_fine_tuned_loss = calculate_test_loss(
    model,
    test,
    tokenizer
)

print("Fine-tuned model loss:", lora_fine_tuned_loss)

Fine-tuned model loss: 1.6571453626025212


In [79]:
base_loss = calculate_test_loss(
    base_model,
    test,
    base_tokenizer
)

print("Base model loss:", base_loss)

Base model loss: 2.113218175221796


In [81]:
print(f"Base model loss:       {base_loss:.4f}")
print(f"Fine-tuned model loss: {lora_fine_tuned_loss:.4f}")
print(f"Loss reduction:        {base_loss - lora_fine_tuned_loss:.4f}")

Base model loss:       2.1132
Fine-tuned model loss: 1.6571
Loss reduction:        0.4561


Training:
3 epochs

Trainable parameters:
1,089,536 / 1,544,803,840
= 0.0705%

Validation loss:
1.5563

Test loss:
Base model       = 2.1132
Fine-tuned model = 1.6593

Test loss reduction:
21.5%

QLoRA Section


In [70]:
!pip install -U bitsandbytes
!pip install -U bitsandbytes>=0.46.1

In [71]:
from transformers import BitsAndBytesConfig
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"


qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    device_map="auto"

)

qlora_model = AutoModelForCausalLM.from_pretrained(checkpoint,quantization_config = qlora_config)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [72]:
from peft import prepare_model_for_kbit_training

qlora_model = prepare_model_for_kbit_training(qlora_model)

In [73]:
from peft import get_peft_model,LoraConfig

qlora_peft_config = LoraConfig(
    target_modules = ["q_proj","v_proj"],
    task_type = "CAUSAL_LM",
    r = 8,
    lora_alpha = 16,
    lora_dropout= 0.05
)

qlora_peft_model = get_peft_model(qlora_model,qlora_peft_config)
qlora_peft_model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [74]:
from trl import SFTConfig,SFTTrainer

sft_config = SFTConfig(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    # Sequence length
    max_length=512,

    # Evaluation
    eval_strategy="epoch",

    # Saving
    save_strategy="epoch",

    # Train only on assistant responses
    assistant_only_loss=True,

    # Optional
    logging_steps=10,
    report_to="none",
)

trainer = SFTTrainer(model=qlora_peft_model,args=sft_config,train_dataset=train_dataset,eval_dataset=valid_dataset)

trainer.train()

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.795989,1.719571,1.551978,18374.000000,0.559681
2,1.603413,1.618935,1.548817,36748.000000,0.585624
3,1.536205,1.593110,1.520821,55122.000000,0.587397


TrainOutput(global_step=75, training_loss=1.670069637298584, metrics={'train_runtime': 374.5413, 'train_samples_per_second': 1.602, 'train_steps_per_second': 0.2, 'total_flos': 486498617659392.0, 'train_loss': 1.670069637298584, 'epoch': 3.0})

In [75]:
inputs = tokenizer(prompt, return_tensors="pt").to(qlora_peft_model.device)

outputs = qlora_peft_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("QLoRA MODEL ANSWER:")
print(generated)

print("\nREFERENCE ANSWER:")
print(reference_answer)

QLoRA MODEL ANSWER:
The drain current increases exponentially with increasing VDS, until it reaches saturation at some maximum value determined by the device's threshold voltage and channel length modulation effects.

REFERENCE ANSWER:
Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.


In [82]:
qlora_fine_tuned_loss = calculate_test_loss(
    qlora_peft_model,
    test,
    tokenizer
)

print("Fine-tuned qlora-model loss:", qlora_fine_tuned_loss)

Fine-tuned qlora-model loss: 1.7316660406961017


In [83]:
import math

base_ppl = math.exp(base_loss)
lora_ppl = math.exp(lora_fine_tuned_loss)
qlora_ppl = math.exp(qlora_fine_tuned_loss)

print(f"Base Perplexity:  {base_ppl:.4f}")
print(f"LoRA Perplexity:  {lora_ppl:.4f}")
print(f"QLoRA Perplexity: {qlora_ppl:.4f}")

Base Perplexity:  8.2748
LoRA Perplexity:  5.2443
QLoRA Perplexity: 5.6501
